# Dimensionality Reduction

## Previous work 

The data set `data/EstrogenReceptorStatus_Test.csv` ,previously explored, has 162 predictors or characteristics, for that we applied 2 feature selection algorithms, this gave us around ~33 predictors selecting the 20 percentile of the most informative predictors sleected with ANOVA and Mutual information methods.


## Dimensionality Reduction
As we have such less predictors in our data set we are now tryng to view the distribution of those in 4 methods that use different approximations, that try to project the high dimensionality of the data in less dimensions here we explore 2 dimensions projection using 

In [ ]:
import os
import pandas as pd
import numpy as np
import umap
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import linalg
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE

if "activities" in (cwd := Path.cwd()).parts or "src" in cwd.parts:
    folder = "activities" if "activities" in cwd.parts else "src"
    root = cwd.parents[len(cwd.parts) - 1 - cwd.parts.index(folder)]
    os.chdir(root)
    print(f"Working in: {Path.cwd()}")
else:
    print(f"Working in {cwd}")
    
base_pth=cwd
scaler = StandardScaler()

In [ ]:
#chargue the data 
data_dic={
    "ANOVA":{
        "Train":pd.read_csv(os.path.join(cwd, "results/feature_selection", "anova_Train.csv"), index_col=0),
        "Test":pd.read_csv(os.path.join(cwd, "results/feature_selection", "anova_Test.csv"), index_col=0)
    },
    "MI":{
        "Train":pd.read_csv(os.path.join(cwd, "results/feature_selection", "mutual_information_Train.csv"), index_col=0),
        "Test":pd.read_csv(os.path.join(cwd, "results/feature_selection", "mutual_information_Test.csv"), index_col=0)
    },
    "Raw":{
        "Train":pd.read_csv(os.path.join(cwd, "data", "EstrogenReceptorStatus_Train.csv"), index_col=0),
        "Test":pd.read_csv(os.path.join(cwd, "data", "EstrogenReceptorStatus_Test.csv"), index_col=0)
    }
}

#get the labels 
test_labels = pd.read_csv(
    os.path.join(cwd, "data", "EstrogenReceptorStatus_Test_labels.txt"), header=None
)
train_labels = pd.read_csv(
    os.path.join(cwd, "data","EstrogenReceptorStatus_Train_labels.txt"), header=None
)
Labels=pd.concat([
    pd.merge(data_dic["Raw"]["Train"].reset_index(), 
            train_labels, left_index=True, right_index=True
    ),pd.merge(data_dic["Raw"]["Test"].reset_index(), 
            test_labels, left_index=True, right_index=True
    )
]).loc[:,["index",0]].set_index("index").rename_axis(None)


# mapper for the labels
lab_map = pd.Series(Labels.iloc[:, 0])

#plots conf
sns.set_theme(style="whitegrid", context="notebook")
palette = {
    np.int64(0): "#C7007C",
    np.int64(1): "#009503"
}

In [ ]:

#functions

def plot_scatter(df_p:pd.DataFrame, axes:list,col_lab:str="Label", ax=None, Name:str="PCA", var_ex=[], colors_pal=None, elev=22, azim=45):
    """
    Function to make the scater represenattion of the projection over distinct elements of reduction of dimensionality algorithms 
    Arguments: 
        -df_p: data frame that must have the axes of the plots (dimension) as well as the column col_lab with the labels of each sample
        -axes: list with the axes name this must be concordant with the ones in the data frame
        -col_lab: column of the data frame to color the dots in the scatter
        -ax: axis to draw the plot if not provided a new figure is maded 
        -Name: name of the plot 
        -var_ex: array with the variance explained for each component of the reduction
    Returns: 
        -scat_p: figure when a axis is not provided 
        -0: when the axes was provided the plot is drawn in that figure ax
    """
    #chech if the fig is maded 
    if not ax:
        scat_p, ax = plt.subplots(figsize=(10, 10))
    
    #clases present in the df
    unique_class=df_p[col_lab].unique()
    
    if not colors_pal: 
        colors_pal=dict(zip(unique_class, sns.color_palette("deep", len(unique_class))))
    #set the dimentionality of the plot 
    dim=len(axes)
    
    #match with the dimensionality 
    match dim:
        case 2: 
            #make the scatter plot of 2 dimentions 
            sns.scatterplot(
                data=df_p,
                x=axes[0],
                y=axes[1],
                hue="Label",
                style="Label",
                palette=colors_pal,
                s=90,
                alpha=0.85,
                ax=ax,
            )
        case 3:
            #plot the scatter of each class to label in the 3 axes 
            for class_t in unique_class:
                sample_class=df_p[df_p[col_lab]==class_t]
                if not sample_class.empty:
                    ax.scatter(
                        sample_class[axes[0]],
                        sample_class[axes[1]],
                        sample_class[axes[2]],
                        label=class_t,
                        color=colors_pal[class_t],
                        s=55,
                        alpha=0.85,
                        edgecolors="white",
                        linewidth=0.4,
                    )
            ax.view_init(elev=elev, azim=azim)
        case _:
            return 1
    #set the title
    ax.set_title(Name)
    #format the axes 
    for n_ax,ax_p in enumerate(axes): 
        #add the explainde variance
        if any(var_ex):
            match n_ax:
                case 0:
                    ax.set_xlabel(f"{axes[n_ax]} ({var_ex[n_ax]*100:.1f}% var)")
                case 1:
                    ax.set_ylabel(f"{axes[n_ax]}  ({var_ex[n_ax]*100:.1f}% var)")
                case 2:
                    ax.set_zlabel(f"{axes[n_ax]}  ({var_ex[n_ax]*100:.1f}% var)")
    
    if ax:
        return 0
    else:
        return scat_p

def variance_plot(var_exp:list, ax=None, val="Principal component", Name="PCA"):
    if not ax:
        var_plot, ax = plt.subplots(figsize=(15, 10))
    #make the data frame with each variance expained for each component as well as the cumulative one
    df_var = pd.DataFrame({
        "Index": np.arange(1, len(var_exp) + 1),
        "Var_explained": var_exp,
        "Cumulative_var": np.cumsum(var_exp),
    })
   
   #tendency of the marginal variance of each component
    sns.lineplot(
        x=np.arange(len(df_var)),
        y=df_var["Var_explained"],
        marker="o",
        markersize=7,
        linewidth=2.5,
        linestyle="--",
        color="#E32D09",
        label="Marginal var (%)",
        ax=ax,
    )
    #tendency of the cumulative variance
    sns.lineplot(
        x=np.arange(len(df_var)),
        y=df_var["Cumulative_var"],
        marker="o",
        markersize=7,
        linewidth=2.5,
        color="#1F78B4",
        label="Cumulate var (%)",
        ax=ax,
    )
    ax.axhline(90.0, color="crimson", linestyle="--", linewidth=1.3, label="Umbral 90%")
    ax.set_ylim(0, 105)
    ax.set_title(f"Varianza Explicada (Train) - {Name}")
    ax.set_xlabel(val)
    ax.set_ylabel("Porcentaje de Varianza (%)")
    ax.legend(loc="upper left",bbox_to_anchor=(1.02, 1))
    print(f"Achieved cumulative variance with {len(var_exp)} components: {max(df_var['Cumulative_var'])}")
    if ax:
        return 0
    else:
        return var_plot

def make_dim_red_var(data:pd.DataFrame, type="PCA", log_t=False, comp=None, var_cut=90,diff=5, low_c=2,high_c=50 ):
    """
    Function that makes the dimention reduction for PCA and SVD descomposition approach
    it normally finds the number of components needed to explain the 90% of the variance
    but it can be maded with a fixed number of components 
    Arguments: 
        -data: dataframe with the data in the instance x characteriscs format
        -type: type of transformation it can be either PCA or SVD
        -log_t: flag that indicates that has to be done a log2 transformation
        -comp: number of component to calculate if is not ptovided it search the number needed to get 90% var
        -var_cut: the var cut of for the search as a default is 90% 
    
    """
    #ask to make the transformation +
    if log_t:
        data=np.log2(data + 1)
    #scaled the data to make it var 1 and mean 0 
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data)
    #set the starting point in order if a search is gonna be done
    if comp:
        mid_comp=comp
    else:
        mid_comp=round((high_c-low_c)/2)
        err=var_cut
    #start the search as a binary one 
    while err > diff:
        #review if the algorithm has a convergence with no apparent resolution
        if mid_comp == low_c or mid_comp == high_c:
            break
        #make the descomposition 
        match type: 
            case "PCA":
                pca = PCA(n_components=mid_comp, random_state=42)
                pca_fit = pca.fit_transform(data_scaled)
                var_exp= pca.explained_variance_ratio_ * 100
            case "SVD":
                #make the descompsition in the 3 terms 
                U, s, Vt = linalg.svd(data_scaled, full_matrices=False)
                #keep only the deisred components 
                k = min(mid_comp, len(s))
                #keep the component values
                comp_vals = s[:k]
                #also mantain the associated vectors 
                Vt_k = Vt[:k, :]
                #obatin the aproximate variance
                var_exp = ((s**2) / np.sum(s**2))[:k] * 100 
                # finaly the coordinates 
                svd_scores = U[:, :k] * comp_vals  
            case _:
                print("Case not supported, only support SVD and PCA")
                return 1 
        #obtain the cumulative sum 
        cum_var_c=sum(var_exp)   
        #review if we are mading the binary search 
        if comp:
            err=0
            comp_f=comp
        else:
            err=var_cut-cum_var_c
            #assing the cut ofs 
            if err > diff: 
                #if is higher it means that the components need to be higher
                low_c=mid_comp
            elif abs(err) > diff:
                #it means that the components are explaining more variance 
                high_c = mid_comp
            #update the error and the mid_comp
            err=abs(err)
            comp_f=mid_comp
            mid_comp=round((high_c+low_c)/2)

    print(f"The run has found a cumulative variance of {cum_var_c} with {comp_f} components")
    #finally return the results 
    match type: 
        case "PCA":
            return pca, pca_fit, var_exp, scaler
        case "SVD":
            return k, comp_vals, Vt_k, var_exp, svd_scores, scaler

## PCA 

In [ ]:
n_comp = 10

for data_set in data_dic.keys():
    pca_plot, axes = plt.subplots(nrows=1, ncols=3, figsize=(20, 7))
    plots_3d, ax_3d = plt.subplots(nrows=1,ncols=2,figsize=(14, 22),
        subplot_kw={"projection": "3d"},
    )
    for i, data_split in enumerate(data_dic[data_set].keys()):
        # log transforms the metabolites and changes the mean (0) and sd (1)
        data_trans = np.log2(data_dic[data_set][data_split] + 1)

        # fit the PCA transformation only if are the train dataset
        if data_split == "Train":
            pca, pca_fit, var_exp, scaler =make_dim_red_var(
                data_trans, 
                type="PCA",
                diff=2
            )
            var_exp_train=var_exp

        else:
            data_scaled = scaler.transform(data_trans)
            pca_fit = pca.transform(data_scaled)

        # obtain the eigenvalues (10,), explained variance ratio and eigenvectors
        prin_vals = pca.explained_variance_
        var_exp = pca.explained_variance_ratio_
        prin_vec = pca.components_

        df_prin = pd.DataFrame(
            pca_fit,
            index=data_trans.index,
            columns=[f"PC{val+1}" for val in range(pca_fit.shape[1])],
        )
        df_prin["Label"] = df_prin.index.map(lab_map)
        # build the plot tow dimensions
        plot_scatter(df_prin, 
            ["PC1", "PC2"] ,
            var_ex=var_exp, ax=axes[i], 
            Name=f"PCA {data_split} in {data_set}", 
            colors_pal=palette
        )
        plot_scatter(df_prin, 
            ["PC1", "PC2", "PC3"] ,
            ax=ax_3d[i],var_ex=var_exp, 
            Name=f"PCA {data_split} in {data_set}", 
            colors_pal=palette
        )
    print(len(var_exp_train))
    #make the plot of the variance explained
    variance_plot(var_exp_train, axes[2],val="Principal component", Name=f"Explained variance in {data_split}-{data_set}")

    plt.tight_layout()
    plt.show()
        

When fitting the PCA transformation to the train data an then projecting the test values on the founded components whe cannot see a clearly segregation within classes but more important the cumulative variance of the first 10 components get ups to 71.4 porcent of the explianed variance in the ANOVA

In [ ]:
for data_set in data_dic.keys():
    print(f"{'='*50}\nAnalizing {data_set}\n{'='*50}")
    #uynify the data splits
    data_set_w=pd.concat([data_dic[data_set]["Train"], data_dic[data_set]["Test"]])
    #plot figs and axes
    pca_multi, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 7))
    plots_3d, ax_3d = plt.subplots(figsize=(14, 22),
        subplot_kw={"projection": "3d"},
    )
    # log transforms the metabolites and changes the mean (0) and sd (1)
    data_trans = np.log2(data_set_w + 1)
    data_scaled = scaler.fit_transform(data_trans)

    # fit the PCA transformation
    pca_fit = pca.fit_transform(data_scaled)

    # obtain the eigenvalues (10,), explained variance ratio and eigenvectors
    prin_vals = pca.explained_variance_
    var_exp = pca.explained_variance_ratio_
    prin_vec = pca.components_
    df_eig=pd.DataFrame(
        {"Component":[f"PC{val+1}" for val in range(pca_fit.shape[1])] , 
         "Value": var_exp
        }
    )
    
    df_prin = pd.DataFrame(
            pca_fit,
            index=data_trans.index,
            columns=[f"PC{val+1}" for val in range(pca_fit.shape[1])],
    )
    df_prin["Label"] = df_prin.index.map(lab_map)

    # build the plot
    plot_scatter(df_prin, 
                 ["PC1", "PC2"], 
                 var_ex=var_exp,ax=axes[0], 
                 Name=f"Principal component in {data_set}",
                 colors_pal=palette
    )
    plot_scatter(df_prin, 
                 ["PC1", "PC2", "PC3"] ,
                 ax=ax_3d,var_ex=var_exp, 
                 Name=f"PCA in {data_set}", 
                 colors_pal=palette
    )
    
    variance_plot(var_exp*100, axes[1], val="Principal component", Name=f"Variance explained in {data_set}")

    plt.tight_layout()
    plt.show()

We can not see a clear separation of the labels in the two dimensionality reducttion space we made using the principal values this trend is in both projection with and withour segregating the data by their data split.

## SVD

In [ ]:
n_comp = 10
for data_set in data_dic.keys():
    svd_plot, axes = plt.subplots(nrows=1, ncols=3, figsize=(20, 7))
    plots_3d, ax_3d = plt.subplots(nrows=1,ncols=2,figsize=(14, 22),
            subplot_kw={"projection": "3d"},
    )
    for i, data_split in enumerate(data_dic[data_set].keys()):
        # log transforms the metabolites and changes the mean (0) and sd (1)
        data_trans = np.log2(data_dic[data_set][data_split] + 1)
        if data_split == "Train":
            print(f"{'='*50}\nAnalizing {data_split}-{data_set}\n{'='*50}")
            
            k, s_k, Vt_k, var_exp ,svd_scores,scaler=make_dim_red_var(
                data_trans, 
                type="SVD",
                diff=2
            )
            var_exp_train=var_exp
        else:
            # Poject the values in the founded vectors 
            data_scaled = scaler.transform(data_trans)
            svd_scores = data_scaled @ Vt_k.T

        # obtain the eigenvalues (10,), explained variance ratio and eigenvectors
        sv_cols = [f"SV{val+1}" for val in range(k)]
        df_prin = pd.DataFrame(svd_scores, index=data_trans.index, columns=sv_cols)
        df_prin["Label"] = df_prin.index.map(lab_map)

        # build the plot
        plot_scatter(df_prin, ["SV1", "SV2"], 
                     var_ex=var_exp_train/100,ax=axes[i], 
                     Name=f"SV1 vs SV2 in {data_split}-{data_set}",
                     colors_pal=palette
        )
        plot_scatter(df_prin, 
                     ["SV1", "SV2","SV3"] ,
                     ax=ax_3d[i],var_ex=var_exp, 
                     Name=f"PCA {data_split} in {data_set}", 
                     colors_pal=palette
        )
        
    #make the plot of the variance explained
    variance_plot(var_exp_train, axes[2], val="Single Value",Name=f"Expelained variance in {data_split}-{data_set}")

    plt.tight_layout()
    plt.show()

## t-SNE

In [ ]:
perplexity_vals=[5,15,30]

num_compar=len(perplexity_vals)

for data_set in data_dic.keys():
    svd_plot, axes = plt.subplots(nrows=num_compar, ncols=2, figsize=(20, 18))
    plots_3d, ax_3d = plt.subplots(nrows=num_compar,ncols=2,figsize=(14, 22),
        subplot_kw={"projection": "3d"},
    )
    for i, data_split in enumerate(data_dic[data_set].keys()):
        if data_split == "Train":
            print(f"{'='*50}\nAnalizing {data_split}-{data_set}\n{'='*50}")
        #overr all the perplexity values 
        for j in range(0,num_compar):
            data_trans = np.log2(data_dic[data_set][data_split] + 1)
            perp = perplexity_vals[j]
            tsne = TSNE(
                n_components=3,
                perplexity=perp,
                init="random",
                learning_rate="auto",
                random_state=42,
            )
            tsne_coords = tsne.fit_transform(data_trans)

            df_tsne = pd.DataFrame(tsne_coords, 
                index=data_trans.index, 
                columns=["tSNE1", "tSNE2", "tSNE3"]
            )
            df_tsne["Label"] = df_tsne.index.map(lab_map)
            #2d plots
            plot_scatter(df_tsne, ["tSNE1", "tSNE2"], 
                ax=axes[j][i], 
                Name=f"tSNE1 vs tSNE2 in {data_split}-{data_set} perplexity={perp}", 
                colors_pal=palette
            )
            #3d plots
            plot_scatter(df_tsne, 
                ["tSNE1", "tSNE2", "tSNE3"],
                ax=ax_3d[j][i], 
                Name=f"tSNE1 vs tSNE2 vs tSNE3 in {data_split}-{data_set} perplexity={perp}", 
                colors_pal=palette
            )
    plt.tight_layout()
    plt.show()
            

## UMAP

In [ ]:
for data_set in data_dic.keys():
    svd_plot, axes = plt.subplots(nrows=1, ncols=2, figsize=(20, 10))
    plots_3d, ax_3d = plt.subplots(nrows=1,ncols=2,figsize=(14, 22),
            subplot_kw={"projection": "3d"},
    )
    for i, data_split in enumerate(data_dic[data_set].keys()):
        data_trans = np.log2(data_dic[data_set][data_split] + 1)
            
        if data_split == "Train":
            print(f"{'='*50}\nAnalizing {data_split}-{data_set}\n{'='*50}")
            umap_model = umap.UMAP(
                n_components=3,
                n_neighbors=30,
                min_dist=0.15,
                metric="euclidean",
                random_state=42,
            )
            umap_coords = umap_model.fit_transform(data_trans)
        else:
            umap_coords = umap_model.transform(data_trans)

        df_umap = pd.DataFrame(umap_coords, 
            index=data_trans.index, 
            columns=["UMAP1", "UMAP2", "UMAP3"]
        )
        df_umap["Label"] = df_umap.index.map(lab_map)
        plot_scatter(df_umap, ["UMAP1", "UMAP2"], 
            ax=axes[i], 
            Name=f"UMAP1 vs UMAP2 in {data_split}-{data_set}", 
            colors_pal=palette
        )
        plot_scatter(df_umap, 
            ["UMAP1", "UMAP2", "UMAP3"] ,
            ax=ax_3d[i],var_ex=var_exp, 
            Name=f"UMAP1 vs UMAP2 vs UMAP3 {data_split} in {data_set}", 
            colors_pal=palette
        )
    plt.tight_layout()
    plt.show()